# Building a Data Analytics Bedrock Agent

## Introduction

Welcome to this hands-on workshop where you'll learn how to build a powerful SQL assistant using Amazon Bedrock Agents. In this lab, you'll create an agent that translates natural language questions into SQL queries for data analysis, specifically focused on a returns and recommerce business scenario.

This workshop will guide you through:
- Creating a Bedrock Agent with Claude 3.5 Haiku
- Adding database interaction capabilities via Action Groups
- Implementing file operations for saving reports and analysis
- Testing your agent with natural language queries
- Managing the complete agent lifecycle

By the end of this lab, you'll have built a fully functional AI assistant that can help business users analyze data without needing to write SQL code themselves. This pattern can be adapted for various business domains and database systems beyond the returns and recommerce example used here.

Let's get started!

### Prerequisites

Please ensure that this notebook is launched in a non-production/burner Isengard/Conduit account.

Before starting this lab, you must follow the instructions here to enable access to all foundation models on Amazon Bedrock. Access will be granted near-instantaneously for internal accounts. We will specifically be working with the Anthropic Claude 3.5 Haiku model.
- https://docs.aws.amazon.com/bedrock/latest/userguide/getting-started.html#getting-started-model-access


### Import libraries, helper functions, and initialize clients

In this section, we set up our environment by importing necessary Python libraries and initializing AWS service clients that we'll need throughout the workshop.

In [ ]:
import boto3
import sqlite3
import time
import uuid
import pandas as pd
import pprint
import logging
import json
from IPython.display import display, HTML
from agents import AgentsForAmazonBedrock
from generic_helper import invoke_agent

# Set up logging with a simple format
logging.basicConfig(level=logging.INFO, format='%(message)s')
logger = logging.getLogger(__name__)

# Initialize session/clients for boto3
session = boto3.session.Session()
region = session.region_name
account_id = session.client('sts').get_caller_identity()['Account']
s3_client = boto3.client("s3",region_name=region)
bedrock = boto3.client("bedrock",region_name=region)
bedrock_runtime = boto3.client("bedrock-runtime",region_name=region)
bedrock_agent_client = boto3.client("bedrock-agent",region_name=region)
bedrock_agent_runtime_client = boto3.client("bedrock-agent-runtime",region_name=region)

In the above code block, we added:

- Standard libraries: For data handling (pandas), database operations (sqlite3), logging, and utility functions
- AWS SDK: boto3 for interacting with AWS services
  - AWS clients: Configured for S3, Bedrock, and Bedrock Agent services in your current region
- Custom helper modules:
  - AgentsForAmazonBedrock: Simplifies agent creation and management
  - invoke_agent: Helper function to test our agent with prompts

All these components provide the foundation for creating and interacting with our Bedrock Agent throughout the workshop.

### Create the Amazon Bedrock Agent

In this section, we create the core Amazon Bedrock Agent that will serve as our SQL assistant.

- **Agent Definition**: We give our agent a name, description, and detailed instructions that shape its behavior and capabilities
  
- **Agent Instructions**: This is the "system prompt" that guides the agent's behavior and defines:
  - Its specific role as a SQL assistant for Returns & ReCommerce
  - How it should interact with users and respond to questions
  - The business domain focus areas (returns, refurbished products, etc.)
  - Output formatting requirements for SQL queries

- **Foundation Model**: We're using Claude 3.5 Haiku, which offers a good balance of capability and cost-efficiency - and also has enough service quota in internal accounts

- **Creation Process**: The code checks if the agent already exists before creating a new one, and includes proper error handling

In [ ]:
agents_handler = AgentsForAmazonBedrock()
agent_name = "SQLWorkshopAgent"
agent_description = "A text-to-SQL agent that helps convert natural language questions into SQL queries for a simplified database."
agent_instructions = """You are a SQL Assistant for Amazon Returns & ReCommerce (RR). Your purpose is to help users analyze data by converting natural language questions into SQL queries.
When responding to user questions:
1. Understand what data they're asking about related to returns, refurbished products, or recommerce operations
2. Use your tools to explore the database structure when needed
3. Generate appropriate SQL queries for SQLite that answer their questions
4. Explain your queries in simple business terms
5. Format SQL with proper indentation and keyword capitalization
Focus on helping users understand:
- Return rates and reasons across marketplaces
- Product performance after returns
- Customer experience metrics
- Pricing and cost analysis for refurbished products
- Profitability comparisons across product conditions
If a user's request is ambiguous, ask clarifying questions before generating SQL.
Provide only factual information based on the database content.
Always verify your queries are correctly formatted for SQLite before presenting them.
"""
# Specify the foundation model to use
foundation_model = "us.anthropic.claude-3-5-haiku-20241022-v1:0"
model_ids = [foundation_model]

# Check if the agent already exists
existing_agent_id = agents_handler.get_agent_id_by_name(agent_name)
if existing_agent_id:
    print(f"Agent '{agent_name}' already exists with ID: {existing_agent_id}")
    agent_id = existing_agent_id
else:
    try:
        # Create the agent
        agent_id = agents_handler.create_agent(
            agent_name=agent_name,
            agent_description=agent_description,
            agent_instructions=agent_instructions,
            model_ids=model_ids
        )
        print(f"Created new agent '{agent_name}' with ID: {agent_id}")
        
        # Wait for agent to be prepared
        print("Preparing agent...")
        bedrock_agent_client = agents_handler._bedrock_agent_client
        bedrock_agent_client.prepare_agent(agentId=agent_id)
        
        # Wait for preparation to complete
        time.sleep(10)
        print(f"Agent '{agent_name}' is ready")
    except Exception as e:
        print(f"Error creating agent: {e}")
        agent_id = agents_handler.get_agent_id_by_name(agent_name)
        if agent_id:
            print(f"Using existing agent with ID: {agent_id}")
        else:
            print("Failed to create or find agent")
            agent_id = None

print(f"Agent ID: {agent_id}")

We immediately test the newly created agent with a simple prompt to verify it's working properly. This initial test confirms that our agent is deployed and responsive before we add more advanced capabilities. The agent can't do much yet because it has no access to any tools.

In [ ]:
# Test the agent
response = invoke_agent(
    bedrock_agent_runtime_client=bedrock_agent_runtime_client,
    agent_id=agent_id,
    prompt="Hello! Please tell me a little bit about yourself and what you can do.", 
    enable_trace=True
)

### Give the agent tools to interact with database via Bedrock Agent Action Groups

In this section, we extend our agent's capabilities by adding tools to interact with a SQLite database. This SQLite database has access to some non-production data from our R&R test tables. These tools allow the agent to explore database structure and execute queries based on natural language requests.

In [ ]:
# Define function schemas for SQLite helper functions
sqlite_functions = [
    {
        'name': 'list_tables',
        'description': 'Lists all available tables in the SQLite database',
        'parameters': {}
    },
    {
        'name': 'describe_table_schema',
        'description': 'Gets column names, data types, and constraints for a specific table',
        'parameters': {
            "table_name": {
                "description": "Name of the table to describe",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'get_sample_rows',
        'description': 'Gets 10 sample rows of data from a table',
        'parameters': {
            "table_name": {
                "description": "Name of the table to get sample rows",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'run_sql_query',
        'description': 'Executes SQL statements against the database and returns results',
        'parameters': {
            "query": {
                "description": "SQL query to execute",
                "required": True,
                "type": "string"
            }
        }
    }
]

# Add the action group with ROC to the agent
action_group_name = "SQLiteTools"
action_group_description = "Tools for interacting with SQLite database including listing tables, describing schemas, and running queries"

try:
    # First check if agent exists
    if agent_id:
        # Add the action group with return of control
        agents_handler.add_action_group_with_roc(
            agent_id=agent_id,
            agent_functions=sqlite_functions,
            agent_action_group_name=action_group_name,
            agent_action_group_description=action_group_description
        )
        print(f"Successfully added SQLite tools action group to agent {agent_name}")
    else:
        print("Agent ID not available. Please create the agent first.")
except Exception as e:
    print(f"Error adding action group: {e}")

In the above code block, we added:
- **Function Definitions**: We define four database tools that give our agent the ability to:
  - Discover available tables in the database
  - Understand table schemas to know what fields are available
  - View sample data to understand content formats
  - Execute SQL queries to answer user questions

- **Action Group Creation**: We organize these tools into a logical "SQLiteTools" action group with a descriptive name and purpose

- **ROC Implementation**: The tools use Return of Control (ROC), meaning the agent can call these functions, get results, then continue its reasoning process with the new information. The logic for what these functions do is defined in `sqlite_helper.py`

Now, we test the enhanced agent with a business question that requires database interaction, showing how the agent now combines natural language understanding with data manipulation capabilities.

In [ ]:
# Test with an analytical question
response = invoke_agent(
    bedrock_agent_runtime_client=bedrock_agent_runtime_client,
    agent_id=agent_id,
    session_id="YOUR_CONVERSATION_ID", # Keep this the same to continue the conversation and ask follow up questions
    prompt="What are the most returned ASINs for march 2025? Fetch the monthly data for March",
    enable_trace=True
)

After running the above code block, you should see the agent leveraging its new tools to analyze the database and formulate an answer. The trace output reveals the agent's step-by-step reasoning process. It should be something like:

1. First, the agent calls the `list_tables` function to discover what tables are available in the database
2. After seeing the available tables, it determines which tables might contain return information
3. The agent then calls `describe_table_schema` on the relevant table to understand its structure
4. Based on the schema, it identifies relevant columns for dates, product identifiers, and metrics
5. Finally, it uses `run_sql_query` to execute an appropriate SQL query to find the most returned ASINs for the specified time period

This demonstrates how the agent autonomously explores the database, understands its structure, and builds appropriate queries - all from a simple natural language request. The agent combines these database tools with its language understanding capabilities to deliver a complete answer without requiring the user to know SQL or database schemas.

### Attach a File Reader/Writer Action Group

In this section, we further enhance our agent by adding file operation capabilities. This allows the agent to save its analyses as reports, log output for future reference, or read existing files for context.

In [ ]:
# Define function schemas for simplified file operations
file_functions = [
    {
        'name': 'read_file',
        'description': 'Reads content from a file',
        'parameters': {
            "file_path": {
                "description": "Path to the file to read, e.g. 'notes.txt', 'query.txt', or 'results.txt'",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'write_file',
        'description': 'Writes content to a file (creates or overwrites)',
        'parameters': {
            "file_path": {
                "description": "Path to the file to write, e.g. 'notes.txt', 'query.txt', or 'results.txt'",
                "required": True,
                "type": "string"
            },
            "content": {
                "description": "Content to write to the file",
                "required": True,
                "type": "string"
            }
        }
    },
    {
        'name': 'list_files',
        'description': 'Lists all files in a directory',
        'parameters': {
            "directory_path": {
                "description": "Path to the directory to list files from, use empty string '' to list all files",
                "required": True,
                "type": "string"
            }
        }
    }
]

# Add the simplified file operations action group with ROC to the agent
file_action_group_name = "FileOperations"
file_action_group_description = "Tools for reading and writing files, and listing directory contents"

try:
    # First check if agent exists
    if agent_id:
        # Add the action group with return of control
        agents_handler.add_action_group_with_roc(
            agent_id=agent_id,
            agent_functions=file_functions,
            agent_action_group_name=file_action_group_name,
            agent_action_group_description=file_action_group_description
        )
        print(f"Successfully added simplified File Operations action group to agent {agent_name}")
    else:
        print("Agent ID not available. Please create the agent first.")
except Exception as e:
    print(f"Error adding file operations action group: {e}")

In the above code block, we defined:

- **File Operation Tools**: We define three essential file manipulation functions that enable our agent to:
  - Read from existing files to incorporate previous analysis or data
  - Write analysis results, reports, or logs to files for persistence
  - List files in directories to discover available resources

- **Report Generation**: With these tools, the agent can now create comprehensive reports based on SQL query results, saving analyses for future reference

- **Persistence Layer**: These capabilities provide a persistence layer so insights aren't lost after the conversation ends

- **Extensibility**: This pattern can be extended to integrate with Amazon internal document systems like Quip, allowing the agent to read from and write to organizational knowledge repositories

Now we will test this by continuing our above conversation. We invoke the agent with the SAME `session_id` and ask it to draft a comprehensive report.

In [ ]:
!sudo chmod 777 . # Give directory read/write permissions
# Ask a follow up question to have the agent write a comprehensive report
response = invoke_agent(
    bedrock_agent_runtime_client=bedrock_agent_runtime_client,
    agent_id=agent_id,
    session_id="YOUR_CONVERSATION_ID", # Keep this the same to continue the conversation and ask follow up questions
    prompt=(
        "Do some additional return rate analysis on an item and product category "
        "level and write a comprehensive report in ./report.txt file"
    ),
    enable_trace=True
)

In [ ]:
# View the results in agent_workbook directory
!ls ./agent_workbook/   # List all files in agent directory
!cat ./agent_workbook/report.txt   # Open report.txt file

After running this code, the agent will create a report file in the `./agent_workbook` directory, containing a comprehensive analysis based on the database information it accessed earlier. By maintaining the same session ID, the agent remembers the context from previous interactions, including which tables it explored and what data was relevant.

These file operation tools complement the database tools we added earlier, creating a more versatile agent that can not only query data but also save and share its findings. This is particularly valuable for data analysis workflows where results often need to be documented and distributed.

These capabilities could be extended to integrate with document management systems, collaboration tools, or other Amazon internal services, allowing the agent to become a seamless part of existing business processes.

### Delete all created resources

In this final section, we clean up the resources created during the workshop to avoid unnecessary costs and maintain a tidy AWS environment.

In [ ]:
agents_handler.delete_agent(agent_name, delete_role_flag=True)

After completing the workshop, it's important to clean up the resources we've created to avoid ongoing charges. The code above deletes our Bedrock Agent and its associated IAM roles.

This cleanup step is an essential best practice when working with cloud resources, especially in a learning environment. The `delete_role_flag=True` parameter ensures that any IAM roles created for the agent are also removed, eliminating potential security concerns from lingering permissions.

For production deployments, you'd want to maintain these resources and implement proper monitoring and maintenance procedures. However, for workshop purposes, cleaning up afterward helps manage costs and keep your AWS account organized.